<!-- GENERATED from diff-hist/docs/public/m_lab_calibration_dashboard.md by tools/gen_public_docs.py — edit the source, not here. -->

# The M-Lab Calibration Dashboard

Identifies potentially uncalibrated M-Lab servers by comparing measurement
distributions across nearby sites: if a server is well calibrated, some other
server should give the same result for at least one client ISP in the region.
Low scores are best, high scores indicate problems.

This version is not suitable for general use because it is prone to both false positive
and false negative results.

The algorithm scans the selected sites for triplets — a target site, a benchmark
site, and a client ISP — where both sites give similar measures of that ISP.
Target sites with high scores (ratio ≫ 1.0 or KS distance ≫ 0.0) have no matching
site and are suspect for calibration problems. Poor results always need to be
checked with other methods (e.g. Regional Details); since we have no ground
truth, the judgement is somewhat subjective.

For information about Differential Histograms and how they expose anomalies in Internet mid-paths
see the **[project overview](https://annealing.mattmathis.net/differential-histograms/)**.

See **[complete](https://annealing.mattmathis.net/differential-histograms/m_lab_calibration_dashboard)** The M-Lab Calibration Dashboard documentation.

In [ ]:
# --- Setup ---
import os, sys, json
from datetime import datetime, date, time, timedelta, timezone

# Locate the repo root (directory containing `converter/`) regardless of where
# Voila/Jupyter is launched from.
_root = os.path.abspath(os.getcwd())
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, "converter")):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

import ipywidgets as widgets
import plotly.graph_objects as go
import pandas as pd
from IPython.display import display, HTML, Markdown

from converter import query_builder as qb, runtime as rt
from converter.widget_builder import Controls

client = rt.bq_client()

# Dashboard variable metadata baked in at conversion time.
VARIABLES = json.loads(r"""
[
  {
    "name": "organization",
    "type": "query",
    "label": "Organization",
    "description": "M-Lab hosting organization.",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "All orgs", "value": ".*" }
    ],
    "current": { "value": ".*" },
    "query_sql": "SELECT text, value FROM ( SELECT 'All orgs' AS text, '.*' AS value, 0 AS _sort UNION ALL SELECT DISTINCT  REGEXP_EXTRACT(site, r'ndt-[a-z0-9]+-[a-z0-9]+\\.([a-z-]+)\\.') AS text,  REGEXP_EXTRACT(site, r'ndt-[a-z0-9]+-[a-z0-9]+\\.([a-z-]+)\\.') AS value,  1 AS _sort FROM `mlab-collaboration.mm_preproduction.cached_metadata` WHERE REGEXP_EXTRACT(site, r'ndt-[a-z0-9]+-[a-z0-9]+\\.([a-z-]+)\\.') IS NOT NULL) ORDER BY _sort, text"
  },
  {
    "name": "method",
    "type": "custom",
    "label": "",
    "description": "Select processing stack version",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "cached", "value": "cached" },
      { "text": "live", "value": "live" }
    ],
    "current": { "value": "cached" },
    "query_sql": "cached, live, experimental"
  },
  {
    "name": "field",
    "type": "custom",
    "label": "",
    "description": "Select the 4th data column.  (Linear fields don't display correctly.) \nMethod must be set to experimental FIRST.",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "MeanThroughputMbps", "value": "MeanThroughputMbps" },
      { "text": "uploadMeanThroughputMbps", "value": "uploadMeanThroughputMbps" },
      { "text": "LossRate", "value": "LossRate" }
    ],
    "current": { "value": "MeanThroughputMbps" },
    "query_sql": "MeanThroughputMbps, uploadMeanThroughputMbps, LossRate"
  },
  {
    "name": "radius",
    "type": "custom",
    "label": "Radius (kM)",
    "description": "Radius from the anchor metro for selecting servers",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "1", "value": "1" },
      { "text": "100", "value": "100" },
      { "text": "500", "value": "500" }
    ],
    "current": { "value": "100" },
    "query_sql": "1, 100, 500"
  },
  {
    "name": "ISPcount",
    "type": "custom",
    "label": "",
    "description": "Number of ISPs to include when searching for calibration triplets",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "5", "value": "5" },
      { "text": "2", "value": "2" },
      { "text": "10", "value": "10" }
    ],
    "current": { "value": "5" },
    "query_sql": "5, 2, 10"
  }
]
""")


In [ ]:
# --- URL parameter presets (webapp mode) ---
# Voila injects the request query string into os.environ["QUERY_STRING"] before
# executing the notebook.  get_query_string() also handles the preheat-kernel
# case (blocks until the request arrives).  Falls back gracefully in plain
# Jupyter where neither is set.
# For scripted or test overrides, set DASH_PRESETS to a JSON object.
import urllib.parse

url_params = {}
try:
    from voila.utils import get_query_string
    _qs = get_query_string() or ""
    for _k, _vs in urllib.parse.parse_qs(_qs).items():
        url_params[_k] = _vs[0] if len(_vs) == 1 else _vs
except Exception:
    pass

_env = os.environ.get("DASH_PRESETS")
if _env:
    url_params.update(json.loads(_env))

# sites= and ISPs= param handling.
# sites= pre-selects servers by site code (e.g. sites=lga04,lga05).
# ISPs= pre-selects ISPs by AS number (e.g. ISPs=7922,8030).
# If sites= is present but anchor= is not, derive anchor from first site code.
_sites_param = [s.strip() for s in url_params.get('sites', '').split(',') if s.strip()]
_isp_asns    = [s.strip() for s in url_params.get('ISPs',  '').split(',') if s.strip()]
if _sites_param and 'anchor' not in url_params:
    url_params['anchor'] = _sites_param[0][:3]
if _sites_param:
    url_params['region'] = _sites_param


In [ ]:
# --- Dashboard controls (dropdowns; query-backed ones are chained) ---

# End date + duration — ignored when method=cached.
# End defaults to the most recent Sunday (UTC); duration defaults to 7 days.
_today_utc  = datetime.now(timezone.utc).date()
_days_back  = (_today_utc.weekday() + 1) % 7
_end_date   = _today_utc - timedelta(days=_days_back)
w_to       = widgets.DatePicker(value=_end_date, description='End (UTC)',
                                style={"description_width": "90px"})
w_duration = widgets.Dropdown(
    options=[("1 day", 1), ("7 days", 7), ("28 days", 28), ("30 days", 30)],
    value=7, description='Duration',
    style={"description_width": "90px"},
)
w_from = None

# Date row: start/end range (exp) or end + duration (otherwise). Empty when the
# flavor has no date pickers.
if w_from is not None:
    _date_row = widgets.HBox([w_from, w_to], layout=widgets.Layout(margin='2px 0'))
elif w_to is not None and w_duration is not None:
    _date_row = widgets.HBox([w_to, w_duration], layout=widgets.Layout(margin='2px 0'))
else:
    _date_row = widgets.HTML('')

# The method (or methodsrc, in exp) selector drives date-picker visibility; the
# date row is spliced into the controls column right after it.
_method_var = 'methodsrc' if 'methodsrc' in [v['name'] for v in VARIABLES] else 'method'
# Put endDate + duration on one row (duration second) where both exist (fleet).
_var_names = [v['name'] for v in VARIABLES]
_hgroups = [['endDate', 'duration']] if 'endDate' in _var_names and 'duration' in _var_names else []
ctrl = Controls(VARIABLES, client, presets=url_params,
                asn_presets={'ClientISP': _isp_asns} if _isp_asns else None,
                after={_method_var: _date_row}, hgroups=_hgroups)
w_run = widgets.Button(description="Run / Refresh", button_style="primary", icon="play")
_date_label = widgets.HTML('')   # filled from query results after Run

# Hide the date row when the backend token is 'cached'; show it otherwise.
_method_w = ctrl.widgets.get(_method_var)
def _toggle_date_row(*_):
    _is_cached = str(getattr(_method_w, 'value', '')).split('-')[0] == 'cached'
    _date_row.layout.display = 'none' if _is_cached else ''
if _method_w is not None:
    _method_w.observe(_toggle_date_row, names='value')
_toggle_date_row()

# "Extra rows" (if present) is shown only when the selected servers span more
# than one metro (distinct 3-letter IATA prefixes of the site codes).
_extra_w   = ctrl.widgets.get('extra_rows')
_servers_w = ctrl.widgets.get('region')
def _toggle_extra_rows(*_):
    _sel = _servers_w.value if _servers_w is not None else ()
    _metros = {str(s)[:3] for s in _sel}
    _row = getattr(_extra_w, 'widget', _extra_w)
    _row.layout.display = '' if len(_metros) > 1 else 'none'
if _extra_w is not None and _servers_w is not None:
    _servers_w.observe(_toggle_extra_rows, names='value')
    _toggle_extra_rows()


In [ ]:
# --- Calibration panels ---
_DATASET  = "mlab-collaboration.mm_preproduction"
_X_AXIS   = "none"
_BIN_SIZE = 50

out = widgets.Output()


def _diagnostics(ctx):
    rows = [(k, ", ".join(v) if isinstance(v, list) else str(v))
            for k, v in ctx.items()]
    return pd.DataFrame(rows, columns=["variable", "value"])


def render(_=None):
    ctx = ctrl.context()
    method  = ctx.get("method", "cached")
    to_dt   = (datetime.combine(w_to.value, time(), tzinfo=timezone.utc)
               if w_to and w_to.value else datetime.now(timezone.utc))
    from_dt = to_dt - timedelta(days=w_duration.value if w_duration else 7)

    # Org selector is a placeholder wired into the report's region_regex slot
    # until the org-filter backend work lands ('.*' = all).
    _org_val = ctx.get("organization") or ".*"
    region_regex = ".*" if _org_val == ".*" else str(_org_val)

    out.clear_output(wait=True)
    with out:
        try:
            df = rt.run_calibration_report(
                client, method, _X_AXIS, _BIN_SIZE,
                ctx.get("field", "MeanThroughputMbps"),
                from_dt, to_dt,
                region_regex,
                int(ctx.get("radius", 100)),
                int(ctx.get("ISPcount", 5)),
                _DATASET,
            )
        except Exception as exc:
            display(HTML(f"<pre>query failed: {exc}</pre>"))
            df = None

        if df is not None and not df.empty:
            display(Markdown("### Scatter plot of KSdistance and ratio"))
            _ratio_col = "Ratio" if "Ratio" in df.columns else "ratio"
            _scatter_df = df[df[_ratio_col] >= 1.0].copy()
            display(go.FigureWidget(rt.plotly_calibration_scatter(_scatter_df)))

            display(Markdown("### Calibration report"))
            # Drop BCargs (leftover debugging column from an earlier link attempt).
            _table_df = df.drop(columns=[c for c in df.columns if c.lower() == "bcargs"],
                                errors="ignore").copy()
            _bc = next((c for c in _table_df.columns if c.lower() == "breadcrumb"), None)
            if _bc:
                _table_df[_bc] = _table_df[_bc].apply(
                    lambda b: (f'<a href="{rt.breadcrumb_to_url(str(b))}" target="_blank"'
                               f' style="text-decoration:underline">{b}</a>')
                    if str(b).strip() else "")
                _table_df = _table_df.rename(columns={_bc: "Breadcrumb"})
            display(widgets.HTML(
                '<div style="height:500px;overflow:auto">'
                + rt.to_html_sticky(_table_df, index=False, na_rep="", escape=False)
                + '</div>'
            ))

        # Cached date range (~5s) — computed after the report so it renders first.
        if method == "cached":
            _date_label.value = (
                '<div style="font-size:12px;color:grey;margin:2px 0"><b>Cached data:</b> '
                + rt.get_cached_date_range(client, _DATASET) + '</div>')

        _diag_out = widgets.Output()
        with _diag_out:
            display(_diagnostics(ctx))
        _diag_acc = widgets.Accordion(children=[_diag_out])
        _diag_acc.set_title(0, "Selector Diagnostics")
        _diag_acc.selected_index = None
        display(_diag_acc)


w_run.on_click(render)
if url_params:
    render()


In [ ]:
# --- Display the app ---
# _date_row is inserted inside ctrl.box (right after the method selector) by
# Controls(after=...); only the status label, Run button, and output remain here.
display(widgets.VBox([ctrl.box, w_run, out, _date_label]))
